# שיעור 13 - זיכרון סוכן


## הגדרה

פנקס רשימות זה מדגים כיצד לבנות סוכן להזמנת טיולים עם **זיכרון מתמשך** באמצעות **מסגרת סוכני מיקרוסופט** (MAF).

תלמד כיצד סוגים שונים של זיכרון סוכן — זיכרון עבודה, קצר טווח וארוך טווח — משפיעים על האופן שבו הסוכן שומר ומשתמש במידע במהלך שיחות.

**דרישות מוקדמות:**
- פרויקט Azure AI Foundry עם מודל שיחה מופעל (למשל `gpt-4o-mini`).
- מחובר עם Azure CLI — הרץ `az login` בטרמינל שלך.
- `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Azure AI Foundry שלך.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהפעלת.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## סוגי הזיכרון של הסוכן

סוכני בינה מלאכותית יכולים לנצל סוגים שונים של זיכרון, שלכל אחד מהם מטרה מובחנת:

### זיכרון עבודה
שרשור השיחה עצמו — ההודעות שהוחלפו במפגש יחיד. הסוכן יכול להתייחס להודעות מוקדמות באותו השרשור כדי לשמור על עקביות. ב-MAF זה נוצר באמצעות **`agent.create_session()`**, שמחזיר `AgentSession`.

### זיכרון קצר טווח
מידע שמתקיים כל עוד נמשכת המשימה או המפגש אך לא נשמר לצמיתות. לדוגמה, הסוכן עשוי לצבור עובדות במהלך שיחת תכנון מרובת סבבים ולהשתמש בהן כדי לייצר את המסלול הסופי.

### זיכרון לטווח ארוך
העדפות ועובדות שמתקיימות **במהלך מפגשים מרובים**. משתמש חוזר לא אמור לחזור ולספר את המגבלות התזונתיות או סגנון הנסיעה שלו. זיכרון לטווח ארוך בדרך כלל מגובה על ידי מאגר חיצוני — בסיס נתונים, קובץ או אינדקס וקטורי — ומוצג לסוכן דרך כלים.


## זיכרון פעיל עם מפגשים

הצורה הפשוטה ביותר של זיכרון היא מפגש השיחה. כשאתה מעביר את אותו מפגש ( שנוצר דרך `agent.create_session()`) לקריאות עוקבות של `agent.run()`, הסוכן רואה את כל היסטוריית השיחה ויכול לזכור פרטים מוקדמים יותר.

בואו ניצור סוכן נסיעות ונמחיש זיכרון פעיל.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

הסוכן זכר נכון את התקציב כי שתי ההודעות חולקות את אותה סשן. זו **זיכרון עבודה** — קיים רק למשך חיי הסשן.

### מה קורה עם שרשור חדש?

אם ניצור סשן **חדש**, לסוכן אין זיכרון של השיחה הקודמת:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## תבנית זיכרון לטווח ארוך

כדי לזכור העדפות משתמש **בין סשנים**, אנחנו צריכים מאגר נתונים קבוע שחי מחוץ לחוט השיחה. הסוכן ניגש למאגר הזה דרך **כלים** — פונקציות שהוא יכול לקרוא כדי לשמור ולהחזיר מידע.

להלן מיישמים מאגר העדפות פשוט בזיכרון (בייצור היית מגבה את זה עם בסיס נתונים או אינדקס וקטורי) ומציגים אותו ככלים שהסוכן יכול להשתמש בהם.

### ארכיטקטורה
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### תרחיש 1 — משתמש בפעם הראשונה מזמין טיול ליום נישואין

שרה מבקרת לראשונה. הסוכן צריך לשמור את ההעדפות שלה באמצעות הכלים ולהשתמש בהם כדי להמליץ על מלונות.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### תרחיש 2 — שרה חוזרת שבועות לאחר מכן

שרה מתחילה **שרשור חדש לגמרי** (מדמה מושב חדש). זיכרון העבודה ריק, אך מאגר ההעדפות לטווח הארוך עדיין מכיל את המידע שלה. הסוכן צריך לשלוף אותו ולהשתמש בו כדי להתאים אישית את ההמלצות.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## סיכום

בשיעור זה למדתם שלושה סוגי זיכרון של סוכן וכיצד לממש אותם עם Microsoft Agent Framework:

| סוג הזיכרון | מנגנון MAF | אורך חיים |
|---|---|---|
| **פועל** | `agent.create_session()` | שיחה בודדת |
| **טווח קצר** | הקשר מצטבר בתוך שרשור | משימה / מפגש יחיד |
| **טווח ארוך** | מאגר חיצוני שניגשים אליו באמצעות פונקציות `@tool` | בין מפגשים |

### נקודות מפתח
1. **`agent.create_session()`** מספק זיכרון פועל — הסוכן רואה את היסטוריית השיחה המלאה בתוך מפגש.
2. **מפגשים חדשים מאבדים הקשר** — ללא זיכרון לטווח ארוך הסוכן לא יכול לזכור שיחות עבר.
3. **פונקציות `@tool`** גושרות על הפער — הן מאפשרות לסוכן לשמור ולשלוף מידע ממאגר מתמיד.
4. **התאמה אישית משתפרת עם הזמן** — ככל ששומרים יותר העדפות, ההמלצות של הסוכן טובות יותר.

### יישומים בעולם האמיתי
- **שירות לקוחות**: לזכור היסטוריה והעדפות של הלקוחות
- **עוזרים אישיים**: לשמור על הקשר לאורך ימים או שבועות
- **בריאות**: לעקוב אחר מידע והעדפות של מטופלים
- **מסחר אלקטרוני**: קנייה מותאמת אישית בהתבסס על היסטוריה

### צעדים הבאים
- להחליף את מילון הזיכרון בזיכרון מבוסס מסד נתונים או מאגר וקטורים (למשל Azure AI Search)
- להוסיף תוקף לזיכרון עבור מידע רגיש לזמן
- לבנות מערכות רב-סוכניות עם זיכרון משותף
- לחקור את המחברת Cognee לזיכרון מונע על ידי גרף ידע


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון כי תרגומים אוטומטיים עלולים להכיל שגיאות או אי דיוקים. המסמך המקורי בשפת המקור שלו צריך להיחשב כמקור המוסמך. עבור מידע קריטי, מומלץ להיעזר בתרגום מקצועי על ידי אדם. אנו לא נושאים באחריות כלשהי עבור אי הבנות או פרשנויות שגויות הנובעות מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
